In [17]:
!pip install -qU langchain langgraph langchain-google-genai pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.3/235.3 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 3.1 MB/s eta 0:00:00


In [22]:
import os
from google.colab import userdata
from pypdf import PdfReader

from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

# API KEY
os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")

# MODEL
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3
)

# PDF TOOL
@tool
def extract_text_from_pdf(file_path: str) -> str:
    """Read text from a PDF file"""

    try:
        reader = PdfReader(file_path)

        text = ""

        for page in reader.pages:
            content = page.extract_text()

            if content:
                text += content + "\n"

        return text

    except Exception as e:
        return f"Error reading PDF: {str(e)}"

# TOOLS
tools = [extract_text_from_pdf]

# SYSTEM PROMPT
system_instruction = """
You are an AI Universal Assistant Bot.

You help with:
1. Answering questions
2. Summarizing PDFs
3. Writing Python code
4. Generating AI and business ideas

Be concise, smart, and helpful.
"""

# CREATE BOT
bot = create_react_agent(
    llm,
    tools,
    prompt=system_instruction
)

# CHAT FUNCTION
def ask_bot(user_prompt):

    inputs = {
        "messages": [("user", user_prompt)]
    }

    response = bot.invoke(inputs)

    print(response["messages"][-1].content)

/tmp/ipykernel_742/1705424228.py:56: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  bot = create_react_agent(


In [23]:
ask_bot("Give me 3 unique startup ideas combining AI and micro-gardening.")

[{'type': 'text', 'text': "Here are 3 unique startup ideas combining AI and micro-gardening:\n\n1.  **AI-Powered Personalized Micro-Garden Coach:** An AI platform that analyzes a user's local climate, available space, sunlight, and desired produce to recommend optimal micro-garden setups, plant varieties, and personalized care schedules. It could offer real-time advice on watering, nutrient levels, and harvesting based on sensor data from the garden.\n2.  **Automated Pest & Disease Detection for Indoor Micro-Gardens:** A smart camera system with AI image recognition that continuously monitors micro-gardens for early signs of pests, diseases, or nutrient deficiencies. It would alert users immediately, identify the specific problem, and suggest organic, targeted solutions to prevent widespread issues.\n3.  **AI-Optimized Hydroponic/Aeroponic Micro-Farm Units:** Compact, self-contained micro-gardening units that use AI to precisely control environmental factors like pH, EC (electrical con

In [24]:
ask_bot("Summarize the key takeaways from the document located at 'sample_report.pdf' inside a bulleted list.")

[{'type': 'text', 'text': "I'm sorry, but I was unable to find the document at 'sample_report.pdf'. Please double-check the file path and try again.", 'extras': {'signature': 'CsQBAQw51se3btc3glPxLl+gfdxRW5ScjzMBEEHkwHtkrxI5Fp0aaJaTB8tLmNKPCjNal+VKSenAmbZIx3kYktotyw+in0FxpHvqsU3tNSh4aXSkBSL4gIj+2XQty2+TEZKVzQPiUXI5SuVpGmJiaHOBu+Txgu/yrqGLnMdAUNKBoCjZ0mhkJgSq3NhLG06ZAE8AHuRaIwKTdCJDMndKWbpDXBMBW8nXc0KuTQGvfTY5fliR/XMebs4V/Dles6kAx3v9QZjT8g=='}}]
